# Multi-Regional Scenarios with GEMSEO Namespaces

This notebook is a **proof-of-concept** for running multiple regional AeroMAPS scenarios and aggregating results to compute worldwide metrics.

## Objectives

1. Create partitioning data for two regions (France and Germany)
2. Create and run two independent regional AeroMAPS processes
3. Explore GEMSEO namespaces to prefix regional outputs
4. Build a simple aggregation mechanism

## Architecture Overview

```
┌─────────────────────────────────────────────────────────────────┐
│                     Multi-Regional Process                       │
├─────────────────────────────────────────────────────────────────┤
│  ┌─────────────┐  ┌─────────────┐                               │
│  │  Region FR  │  │  Region DE  │   (can be parallelized)      │
│  │  (ns:FR)    │  │  (ns:DE)    │                               │
│  └──────┬──────┘  └──────┬──────┘                               │
│         │                │                                       │
│         └────────┬───────┘                                       │
│                  ▼                                               │
│        ┌──────────────────────┐                                 │
│        │   Aggregation Layer  │                                 │
│        │   (Global Metrics)   │                                 │
│        └──────────────────────┘                                 │
└─────────────────────────────────────────────────────────────────┘
```

## 1. Prepare Regional Data

First, we generate the partitioning input files for both regions using AeroSCOPE data.

In [ ]:
from aeromaps.utils.functions import create_partitioning

# Generate partitioning data for France
create_partitioning(
    file="data/region_france/aeroscope_data.csv", 
    path="data/region_france"
)
print("✓ France partitioning data generated")

# Generate partitioning data for Germany  
create_partitioning(
    file="data/region_germany/aeroscope_data.csv", 
    path="data/region_germany"
)
print("✓ Germany partitioning data generated")

## 2. Run Regional Processes Independently

Let's first run each regional process independently to verify they work.

In [ ]:
%matplotlib widget
from aeromaps import create_process

# Create and run France process
process_france = create_process(configuration_file="data/region_france/config.yaml")
process_france.compute()
print("✓ France process computed")

# Create and run Germany process
process_germany = create_process(configuration_file="data/region_germany/config.yaml")
process_germany.compute()
print("✓ Germany process computed")

### Compare key outputs from both regions

In [ ]:
import pandas as pd

# Extract some key outputs for comparison
years = [2020, 2030, 2040, 2050]

comparison_data = {
    "Year": years,
    "FR - CO2 (Mt)": [process_france.data["vector_outputs"]["co2_emissions_including_energy"].loc[y] / 1e9 for y in years],
    "DE - CO2 (Mt)": [process_germany.data["vector_outputs"]["co2_emissions_including_energy"].loc[y] / 1e9 for y in years],
    "FR - RPK (billion)": [process_france.data["vector_outputs"]["rpk"].loc[y] / 1e9 for y in years],
    "DE - RPK (billion)": [process_germany.data["vector_outputs"]["rpk"].loc[y] / 1e9 for y in years],
}

df_comparison = pd.DataFrame(comparison_data)
df_comparison.set_index("Year", inplace=True)
print("Regional comparison:")
display(df_comparison)

## 3. Explore GEMSEO Namespaces

Now let's explore how GEMSEO namespaces work with AeroMAPS disciplines.

### 3.1 Understanding the Discipline Structure

In [ ]:
# Let's examine the disciplines from one of our processes
print(f"Number of disciplines in France process: {len(process_france.disciplines)}")
print("\nFirst 5 disciplines:")
for i, disc in enumerate(process_france.disciplines[:5]):
    print(f"  {i+1}. {disc.name}")
    print(f"      Inputs: {list(disc.input_grammar.names)[:3]}...")
    print(f"      Outputs: {list(disc.output_grammar.names)[:3]}...")

### 3.2 Applying Namespaces to a Discipline

GEMSEO provides `add_namespace_to_input()` and `add_namespace_to_output()` methods.
Let's test this on a simple discipline.

In [ ]:
from copy import deepcopy

# Get a discipline to experiment with
test_disc = deepcopy(process_france.disciplines[0])

print(f"Discipline: {test_disc.name}")
print(f"\nOriginal outputs: {list(test_disc.output_grammar.names)}")

# Apply namespace to outputs
for output_name in list(test_disc.output_grammar.names):
    test_disc.add_namespace_to_output(output_name, "FR")

print(f"\nNamespaced outputs: {list(test_disc.output_grammar.names)}")

### 3.3 Creating Namespaced Discipline Sets for Each Region

Now let's create a helper function to apply namespaces to all disciplines in a process.

In [ ]:
def apply_namespace_to_disciplines(disciplines, namespace: str):
    """
    Apply a namespace prefix to all inputs and outputs of the given disciplines.
    
    Parameters
    ----------
    disciplines : list
        List of GEMSEO disciplines
    namespace : str
        Namespace prefix to apply (e.g., 'FR', 'DE')
        
    Returns
    -------
    list
        List of disciplines with namespaced I/O
    """
    namespaced_disciplines = []
    
    for disc in disciplines:
        # Deep copy to avoid modifying original
        ns_disc = deepcopy(disc)
        
        # Apply namespace to all outputs
        for name in list(ns_disc.output_grammar.names):
            ns_disc.add_namespace_to_output(name, namespace)
            
        # Apply namespace to all inputs
        for name in list(ns_disc.input_grammar.names):
            ns_disc.add_namespace_to_input(name, namespace)
            
        namespaced_disciplines.append(ns_disc)
        
    return namespaced_disciplines

print("✓ Helper function defined")

In [ ]:
# Apply namespaces to both regions
fr_disciplines = apply_namespace_to_disciplines(process_france.disciplines, "FR")
de_disciplines = apply_namespace_to_disciplines(process_germany.disciplines, "DE")

print(f"Created {len(fr_disciplines)} namespaced disciplines for France")
print(f"Created {len(de_disciplines)} namespaced disciplines for Germany")

# Verify namespacing worked
print(f"\nExample France discipline outputs: {list(fr_disciplines[0].output_grammar.names)[:3]}")
print(f"Example Germany discipline outputs: {list(de_disciplines[0].output_grammar.names)[:3]}")

## 4. Create a Simple Aggregation Discipline

Now we create a discipline that takes namespaced regional outputs and aggregates them.

In [ ]:
import numpy as np
from gemseo.disciplines.auto_py import AutoPyDiscipline

# Define variables to aggregate (linear sum)
LINEAR_SUM_VARIABLES = [
    "co2_emissions_including_energy",
    "rpk",
    "ask",
    "rtk",
    "energy_consumption",
]

# Define regions
REGIONS = ["FR", "DE"]


def create_aggregation_function(variables, regions):
    """
    Create an aggregation function dynamically based on variables and regions.
    """
    # Build input parameter string
    params = []
    for var in variables:
        for region in regions:
            # GEMSEO namespaces use ':' separator
            params.append(f"{region}:{var}")
    
    def aggregation_func(**kwargs):
        """Aggregate regional outputs into global metrics."""
        results = {}
        
        for var in variables:
            # Sum values from all regions
            total = None
            for region in regions:
                key = f"{region}:{var}"
                if key in kwargs:
                    value = kwargs[key]
                    if total is None:
                        total = value.copy() if hasattr(value, 'copy') else value
                    else:
                        total = total + value
            
            results[f"global:{var}"] = total
            
        return results
    
    return aggregation_func, params


# Create the aggregation function
agg_func, agg_params = create_aggregation_function(LINEAR_SUM_VARIABLES, REGIONS)
print(f"Aggregation inputs (first 6): {agg_params[:6]}")
print(f"Aggregation outputs: global:{LINEAR_SUM_VARIABLES[0]}, ...")

## 5. Attempt to Build a Combined MDAChain

Now let's try to build an MDAChain that combines both regional discipline sets and the aggregation discipline.

**Note:** This is experimental - we're testing GEMSEO's namespace capabilities.

In [ ]:
from gemseo.mda.mda_chain import MDAChain
from gemseo.core.chains.chain import MDOChain

# First, let's try a simpler approach: 
# Run the existing MDA chains and then aggregate manually

print("Testing manual aggregation of results...")

# Get results from both processes
fr_outputs = process_france.mda_chain.local_data
de_outputs = process_germany.mda_chain.local_data

print(f"\nFrance MDA has {len(fr_outputs)} output variables")
print(f"Germany MDA has {len(de_outputs)} output variables")

In [ ]:
# Manual aggregation of key metrics
import pandas as pd

def aggregate_regional_outputs(processes: dict, variables: list) -> dict:
    """
    Aggregate outputs from multiple regional processes.
    
    Parameters
    ----------
    processes : dict
        Dictionary mapping region names to AeroMAPSProcess instances
    variables : list
        List of variable names to aggregate (linear sum)
        
    Returns
    -------
    dict
        Dictionary with regional (namespaced) and global outputs
    """
    aggregated = {
        "regional": {},
        "global": {}
    }
    
    # Collect regional outputs with namespace
    for region_name, process in processes.items():
        for var in variables:
            if var in process.mda_chain.local_data:
                namespaced_key = f"{region_name}:{var}"
                aggregated["regional"][namespaced_key] = process.mda_chain.local_data[var]
    
    # Compute global aggregates
    for var in variables:
        total = None
        for region_name, process in processes.items():
            if var in process.mda_chain.local_data:
                value = process.mda_chain.local_data[var]
                if total is None:
                    total = value.copy() if hasattr(value, 'copy') else value
                else:
                    total = total + value
        if total is not None:
            aggregated["global"][f"global:{var}"] = total
            
    return aggregated


# Aggregate results
regional_processes = {
    "FR": process_france,
    "DE": process_germany,
}

aggregated_results = aggregate_regional_outputs(
    regional_processes, 
    LINEAR_SUM_VARIABLES
)

print(f"Regional outputs: {len(aggregated_results['regional'])} variables")
print(f"Global outputs: {len(aggregated_results['global'])} variables")

In [ ]:
# Display aggregated results
years = [2020, 2030, 2040, 2050]

print("=" * 60)
print("MULTI-REGIONAL AGGREGATED RESULTS")
print("=" * 60)

for var in ["co2_emissions_including_energy", "rpk", "energy_consumption"]:
    print(f"\n{var.upper()}:")
    print("-" * 40)
    
    # Create comparison dataframe
    data = {"Year": years}
    
    # Regional values
    for region in REGIONS:
        key = f"{region}:{var}"
        if key in aggregated_results["regional"]:
            values = aggregated_results["regional"][key]
            if var == "co2_emissions_including_energy":
                data[f"{region} (Mt)"] = [values[y] for y in years]
            elif var == "rpk":
                data[f"{region} (billion)"] = [values[y] for y in years]
            else:
                data[f"{region}"] = [values[y] for y in years]
    
    # Global value
    global_key = f"global:{var}"
    if global_key in aggregated_results["global"]:
        values = aggregated_results["global"][global_key]
        if var == "co2_emissions":
            data["GLOBAL (Mt)"] = [values[y] for y in years]
        elif var == "rpk":
            data["GLOBAL (billion)"] = [values[y] for y in years]
        else:
            data["GLOBAL"] = [values[y] for y in years]
    
    df = pd.DataFrame(data).set_index("Year")
    display(df)

## 6. Prototype: MultiRegionalProcess Wrapper

Let's create a simple wrapper class that encapsulates the multi-regional workflow.

In [ ]:
from aeromaps import create_process
from aeromaps.utils.functions import create_partitioning
from typing import Dict, List, Optional
import os


class MultiRegionalProcess:
    """
    Prototype wrapper for multi-regional AeroMAPS scenarios.
    
    This class manages multiple regional AeroMAPS processes and provides
    aggregation of results into global metrics.
    """
    
    # Default variables for linear aggregation
    DEFAULT_LINEAR_SUM_VARS = [
        "co2_emissions",
        "cumulative_co2_emissions",
        "rpk",
        "ask", 
        "rtk",
        "energy_consumption",
        "dropin_fuel_consumption",
    ]
    
    def __init__(
        self, 
        regions: Dict[str, dict],
        linear_sum_variables: Optional[List[str]] = None,
    ):
        """
        Initialize the multi-regional process.
        
        Parameters
        ----------
        regions : dict
            Dictionary mapping region identifiers to configuration:
            {
                "FR": {
                    "config_file": "path/to/config.yaml",
                    "aeroscope_file": "path/to/aeroscope.csv",  # optional
                    "data_path": "path/to/data/folder",  # optional
                },
                ...
            }
        linear_sum_variables : list, optional
            Variables to aggregate by linear sum. Defaults to common outputs.
        """
        self.regions = regions
        self.region_names = list(regions.keys())
        self.processes: Dict[str, any] = {}
        self.linear_sum_variables = linear_sum_variables or self.DEFAULT_LINEAR_SUM_VARS
        self.aggregated_results = None
        self._computed = False
        
    def prepare_partitioning(self):
        """Generate partitioning data for all regions that have aeroscope files."""
        for region_name, config in self.regions.items():
            if "aeroscope_file" in config and "data_path" in config:
                print(f"Generating partitioning for {region_name}...")
                create_partitioning(
                    file=config["aeroscope_file"],
                    path=config["data_path"]
                )
        print("✓ Partitioning data prepared for all regions")
        
    def create_processes(self):
        """Create AeroMAPS processes for all regions."""
        for region_name, config in self.regions.items():
            print(f"Creating process for {region_name}...")
            self.processes[region_name] = create_process(
                configuration_file=config["config_file"]
            )
        print(f"✓ Created {len(self.processes)} regional processes")
        
    def compute(self):
        """Compute all regional processes and aggregate results."""
        if not self.processes:
            self.create_processes()
            
        # Compute each regional process
        for region_name, process in self.processes.items():
            print(f"Computing {region_name}...")
            process.compute()
            
        # Aggregate results
        self._aggregate_results()
        self._computed = True
        print("✓ All regional processes computed and aggregated")
        
    def _aggregate_results(self):
        """Aggregate regional outputs into global metrics."""
        self.aggregated_results = {
            "regional": {},
            "global": {}
        }
        
        # Collect regional outputs with namespace
        for region_name, process in self.processes.items():
            for var in self.linear_sum_variables:
                if var in process.mda_chain.local_data:
                    namespaced_key = f"{region_name}:{var}"
                    self.aggregated_results["regional"][namespaced_key] = \
                        process.mda_chain.local_data[var]
        
        # Compute global aggregates (linear sum)
        for var in self.linear_sum_variables:
            total = None
            for region_name, process in self.processes.items():
                if var in process.mda_chain.local_data:
                    value = process.mda_chain.local_data[var]
                    if total is None:
                        total = value.copy() if hasattr(value, 'copy') else value
                    else:
                        total = total + value
            if total is not None:
                self.aggregated_results["global"][f"global:{var}"] = total
                
    def get_regional_output(self, region: str, variable: str):
        """Get a specific output from a region."""
        if not self._computed:
            raise ValueError("Call compute() first")
        return self.processes[region].mda_chain.local_data.get(variable)
    
    def get_global_output(self, variable: str):
        """Get a global aggregated output."""
        if not self._computed:
            raise ValueError("Call compute() first")
        return self.aggregated_results["global"].get(f"global:{variable}")
    
    def summary(self, years: List[int] = None) -> pd.DataFrame:
        """Generate a summary table of key outputs."""
        if not self._computed:
            raise ValueError("Call compute() first")
            
        years = years or [2020, 2030, 2040, 2050]
        
        data = {"Year": years}
        
        # CO2 emissions
        for region in self.region_names:
            key = f"{region}:co2_emissions"
            if key in self.aggregated_results["regional"]:
                values = self.aggregated_results["regional"][key]
                data[f"{region} CO2 (Mt)"] = [values[y] for y in years]
                
        global_co2 = self.aggregated_results["global"].get("global:co2_emissions")
        if global_co2 is not None:
            data["TOTAL CO2 (Mt)"] = [global_co2[y] for y in years]
            
        return pd.DataFrame(data).set_index("Year")


print("✓ MultiRegionalProcess class defined")

### Test the MultiRegionalProcess wrapper

In [ ]:
# Configure regions
regions_config = {
    "FR": {
        "config_file": "data/region_france/config.yaml",
        "aeroscope_file": "data/region_france/aeroscope_data.csv",
        "data_path": "data/region_france",
    },
    "DE": {
        "config_file": "data/region_germany/config.yaml",
        "aeroscope_file": "data/region_germany/aeroscope_data.csv",
        "data_path": "data/region_germany",
    },
}

# Create multi-regional process
multi_process = MultiRegionalProcess(regions=regions_config)

# Prepare partitioning data
multi_process.prepare_partitioning()

# Compute all regions
multi_process.compute()

In [ ]:
# Display summary
print("\nMULTI-REGIONAL SCENARIO SUMMARY")
print("=" * 50)
display(multi_process.summary())

In [ ]:
# Access individual outputs
fr_co2_2050 = multi_process.get_regional_output("FR", "co2_emissions")[2050]
de_co2_2050 = multi_process.get_regional_output("DE", "co2_emissions")[2050]
global_co2_2050 = multi_process.get_global_output("co2_emissions")[2050]

print(f"\n2050 CO2 Emissions:")
print(f"  France:  {fr_co2_2050:.2f} Mt")
print(f"  Germany: {de_co2_2050:.2f} Mt")
print(f"  Total:   {global_co2_2050:.2f} Mt")
print(f"\n  Verification: {(fr_co2_2050 + de_co2_2050):.2f} Mt (sum of regions)")

## 7. Next Steps & Observations

### What we achieved:
1. ✅ Created and ran two independent regional AeroMAPS processes
2. ✅ Explored GEMSEO namespace application to disciplines
3. ✅ Built a simple aggregation mechanism for linear-sum variables
4. ✅ Created a prototype `MultiRegionalProcess` wrapper class

### Key observations:
- GEMSEO namespaces work with `add_namespace_to_input/output()` methods
- Each regional process runs independently (no coupling)
- Linear aggregation (sum) works well for extensive quantities (CO2, RPK, energy)

### For Phase 2 (future work):
1. **GEMSEO-native parallel execution**: Use `MDAChain` with `mdachain_parallelize_tasks=True`
2. **Weighted averages**: Implement for intensive quantities (load factor, efficiency)
3. **Climate model integration**: Run global climate model on aggregated emissions
4. **Configuration-driven aggregation rules**: YAML-based specification of what/how to aggregate
5. **Plotting support**: Multi-regional comparison plots

In [ ]:
# Cleanup for testing
from aeromaps.utils.functions import clean_notebooks_on_tests
clean_notebooks_on_tests(globals())